# Notebook 16 – Complete Preprocessing Workflow


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

rng = np.random.default_rng(21)
n = 120
raw = pd.DataFrame({
    'CustomerID': np.arange(1001, 1001+n),
    'Age': rng.integers(19, 68, n).astype(float),
    'Income': rng.integers(20000, 120000, n).astype(float),
    'City': rng.choice(['Hassan','Mysuru','Bengaluru'], n),
    'Member': rng.choice(['Yes','No'], n),
    'Churn': np.array([1]*18 + [0]*102)
})
raw.loc[[2,11,31], 'Age'] = np.nan
raw.loc[[5,28], 'Income'] = np.nan
raw.loc[9, 'Age'] = -4
raw.loc[17, 'Income'] = 900000  # possible valid extreme; review and cap for this demo
raw.loc[6, 'City'] = None
raw = pd.concat([raw, raw.iloc[[4]]], ignore_index=True)
print('Loaded dataset:', raw.shape)
print(raw.head())

Loaded dataset: (121, 6)
   CustomerID   Age    Income       City Member  Churn
0        1001  33.0   51161.0  Bengaluru    Yes      1
1        1002  57.0   53930.0     Mysuru    Yes      1
2        1003   NaN   91972.0     Mysuru    Yes      1
3        1004  48.0  108240.0  Bengaluru    Yes      1
4        1005  41.0   87690.0  Bengaluru    Yes      1


## Step 1. Load Dataset

We create a realistic small customer dataset inside the notebook. It contains numeric and categorical features, a binary churn target, missing values, a duplicate, an invalid age, and an extreme income. No CSV download is needed.

In [5]:
print("Rows, columns:", raw.shape)

Rows, columns: (121, 6)


## Step 2. Inspect Dataset

Check the first rows, shape, and column information. Inspection identifies obvious quality problems before modeling. It also confirms the target column.

In [7]:
print(raw.head(3))
print(raw.dtypes)

   CustomerID   Age   Income       City Member  Churn
0        1001  33.0  51161.0  Bengaluru    Yes      1
1        1002  57.0  53930.0     Mysuru    Yes      1
2        1003   NaN  91972.0     Mysuru    Yes      1
CustomerID      int32
Age           float64
Income        float64
City           object
Member         object
Churn           int32
dtype: object


## Step 3. Identify Data Types

Age and Income are numerical. City and Member are categorical. CustomerID is an identifier, and Churn is the target. IDs should not be used as numeric predictive features by default.

In [9]:
numeric_features = ["Age", "Income"]
categorical_features = ["City", "Member"]
print("Numeric:", numeric_features, "Categorical:", categorical_features)

Numeric: ['Age', 'Income'] Categorical: ['City', 'Member']


## Step 4. Identify Missing Values

Count missing entries before any imputation. This helps choose appropriate strategies. Missingness can be different across features.

In [11]:
print(raw.isna().sum())

CustomerID    0
Age           3
Income        2
City          1
Member        0
Churn         0
dtype: int64


## Step 5. Handle Missing Values

Missing values are filled inside a fitted pipeline, not using global statistics. Numeric features use median imputation. Categorical features use a constant Unknown label. Training data alone determines learned values.

In [13]:
print("Missing value handling will be fitted inside the training pipeline.")

Missing value handling will be fitted inside the training pipeline.


## Step 6. Detect Duplicates

Check exact repeated rows. Duplicates may bias the model and leak across splits. Verify that the repeated row is not a genuine second transaction.

In [15]:
print("Exact duplicates:", raw.duplicated().sum())

Exact duplicates: 1


## Step 7. Handle Duplicates

Drop only confirmed exact duplicates. Preserve distinct events even when customer IDs repeat. This demonstration contains one deliberately copied row.

In [17]:
data = raw.drop_duplicates().copy()
print("After deduplication:", data.shape)

After deduplication: (120, 6)


## Step 8. Validate Data

Apply a clear business rule to detect impossible ages. A negative age is invalid. The target must be 0 or 1. Other numerical fields should also satisfy domain constraints.

In [19]:
print("Invalid ages:", (~data["Age"].isna() & ~data["Age"].between(18, 100)).sum())
print("Valid target:", data["Churn"].isin([0,1]).all())

Invalid ages: 1
Valid target: True


## Step 9. Detect Outliers

Review extreme values instead of automatically deleting them. An income of 900000 may be genuine. A negative age is a data error. These cases require different decisions.

In [21]:
print("Very high incomes:\n", data.loc[data["Income"] > 200000, ["CustomerID", "Income"]])

Very high incomes:
     CustomerID    Income
17        1018  900000.0


## Step 10. Treat Outliers

The invalid age is changed to missing and imputed later. For this demonstration, an upper income cap is learned from the training data. This policy is illustrative and should be reviewed with domain experts.

In [23]:
data.loc[~data["Age"].isna() & ~data["Age"].between(18,100), "Age"] = np.nan
print("Invalid ages after correction:", (~data["Age"].isna() & ~data["Age"].between(18,100)).sum())

Invalid ages after correction: 0


## Step 11. Encode Categorical Variables

One-hot encoding converts nominal categories into numeric indicators. It avoids imposing a false order on cities. Unknown categories are safely ignored. The encoder is fitted on training rows only.

In [25]:
print("Categories to encode:", categorical_features)

Categories to encode: ['City', 'Member']


## Step 12. Scale Numerical Variables

StandardScaler centers and scales numerical features. It is fitted after imputation. Logistic regression can benefit from scaling. Test rows reuse training statistics.

In [27]:
print("Numerical features to scale:", numeric_features)

Numerical features to scale: ['Age', 'Income']


## Step 13. Handle Class Imbalance

Inspect class counts before deciding whether resampling is needed. We use class_weight=balanced to account for minority-class importance without synthetic samples. Class weights are computed during model fitting on training labels.

In [29]:
print("Class distribution:\n", data["Churn"].value_counts())

Class distribution:
 Churn
0    102
1     18
Name: count, dtype: int64


## Step 14. Perform Feature Selection

Feature selection can remove weak predictors. It must be fitted only on training data. We use SelectKBest after encoding and scaling. Its selected features are then passed to the classifier.

In [31]:
print("Feature selection method: SelectKBest(f_classif, k=4)")

Feature selection method: SelectKBest(f_classif, k=4)


## Step 15. Split Data

Split before fitting any learned preprocessing rules. Stratification helps preserve class proportions. Identifiers are excluded from features. The test set stays unseen until evaluation.

In [33]:
X = data.drop(columns=["CustomerID", "Churn"])
y = data["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (90, 4) Test: (30, 4)


## Step 16. Build Preprocessing Pipeline

Numeric values are imputed, capped using a training-derived limit, and scaled. Categorical values are imputed and one-hot encoded. The resulting features pass through selection and logistic regression. All learned steps are fitted using training data only.

In [35]:
from sklearn.base import BaseEstimator, TransformerMixin
class UpperQuantileCapper(BaseEstimator, TransformerMixin):
    def __init__(self, quantile=0.98): self.quantile = quantile
    def fit(self, X, y=None):
        self.upper_ = np.quantile(np.asarray(X, dtype=float), self.quantile, axis=0)
        return self
    def transform(self, X):
        return np.minimum(np.asarray(X, dtype=float), self.upper_)

num_pipe = Pipeline([('impute', SimpleImputer(strategy='median')),
                     ('cap', UpperQuantileCapper(quantile=0.98)),
                     ('scale', StandardScaler())])
cat_pipe = Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
                     ('encode', OneHotEncoder(handle_unknown='ignore'))])
prep = ColumnTransformer([('num', num_pipe, numeric_features),
                          ('cat', cat_pipe, categorical_features)])
model = Pipeline([('prep', prep), ('select', SelectKBest(score_func=f_classif, k=4)),
                  ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))])
model.fit(X_train, y_train)
print('Pipeline fitted on training data only.')

Pipeline fitted on training data only.


## Step 17. Validate Final Dataset

Transform training and test rows using the fitted preprocessing pipeline. Verify dimensions, missingness, and finite numeric values. Evaluate on held-out test data. Class imbalance means accuracy alone is not enough; check precision and recall.

In [37]:
X_train_ready = model.named_steps['prep'].transform(X_train)
X_test_ready = model.named_steps['prep'].transform(X_test)
X_train_selected = model.named_steps['select'].transform(X_train_ready)
X_test_selected = model.named_steps['select'].transform(X_test_ready)
print('ML-ready train:', X_train_selected.shape, 'test:', X_test_selected.shape)
print('All finite:', np.isfinite(X_train_selected).all(), np.isfinite(X_test_selected).all())
print(classification_report(y_test, model.predict(X_test), zero_division=0))

ML-ready train: (90, 4) test: (30, 4)
All finite: True True
              precision    recall  f1-score   support

           0       1.00      0.58      0.73        26
           1       0.27      1.00      0.42         4

    accuracy                           0.63        30
   macro avg       0.63      0.79      0.58        30
weighted avg       0.90      0.63      0.69        30



## Final Output

`X_train_selected` and `X_test_selected` are the **ML-ready numerical feature matrices**. `y_train` and `y_test` are their corresponding labels. The fitted `model` includes the complete preprocessing, selection, and classifier workflow.

**Why this order?** Confirmed duplicates and impossible ages are handled with fixed business rules before splitting. Imputation, quantile capping, scaling, encoding, and feature selection learn only from training rows, avoiding data leakage. Valid extremes are not automatically deleted. Class imbalance is addressed with class weights rather than blind oversampling.